In [1]:
import pandas as pd
import numpy as np

# === Datei laden ===
df = pd.read_csv("ParetoFront.csv")  # ggf. Pfad anpassen

# === Zielspalten definieren ===
all_objectives = [
    "Driver Violation",
    "Commute Distance",
    "Transport Machines",
    "Transport Attachments",
    "Machines",
    "Workers",
    "Attachments"
]

# === 1. Paretofront aus Transport Attachments & Attachments extrahieren ===
def pareto_front_2d(points):
    points = np.array(points)
    is_efficient = np.ones(points.shape[0], dtype=bool)
    for i, c in enumerate(points):
        if is_efficient[i]:
            is_efficient[is_efficient] = (
                np.any(points[is_efficient] < c, axis=1)
                | np.all(points[is_efficient] == c, axis=1)
            )
            is_efficient[i] = True
    return points[is_efficient]

pareto_points = pareto_front_2d(df[["Transport Attachments", "Attachments"]].values)
df_pareto_attach = pd.DataFrame(
    pareto_points, columns=["Transport Attachments", "Attachments"]
).drop_duplicates()

# === 2. Lösungen erweitern ===
df_expanded = df.drop(columns=["Transport Attachments", "Attachments"]).merge(
    df_pareto_attach, how="cross"
)

# === 3. Finaler Paretofilter (alle Ziele) ===
def pareto_filter_nd(df, objective_cols):
    values = df[objective_cols].values
    is_efficient = np.ones(values.shape[0], dtype=bool)
    for i, v in enumerate(values):
        if is_efficient[i]:
            is_efficient[is_efficient] = (
                np.any(values[is_efficient] < v, axis=1)
                | np.all(values[is_efficient] == v, axis=1)
            )
            is_efficient[i] = True
    return df[is_efficient].reset_index(drop=True)

df_final_pareto = pareto_filter_nd(df_expanded, all_objectives)

# === Insights ===
print("🔍 Insights zur Paretoanalyse\n" + "-"*40)
print(f"📦 Ursprüngliche Lösungen:         {len(df)}")
print(f"🎯 Pareto-Kombinationen (2D):      {len(df_pareto_attach)}")
print(f"🧩 Erweiterte Lösungskombis:       {len(df_expanded)}")
print(f"✅ Nicht-dominierte Endlösungen:   {len(df_final_pareto)}")
print(f"❌ Entfernte (dominierte) Lösungen: {len(df_expanded) - len(df_final_pareto)}\n")

print("📊 Pareto-Kombinationen (Anbaugeräte):")
print(df_pareto_attach.sort_values(["Transport Attachments", "Attachments"]).to_string(index=False))

# === Ergebnis speichern ===
df_final_pareto.to_csv("ParetoFront_filtered.csv", index=False)
print(f"\n💾 Datei gespeichert unter: ParetoFront_filtered.csv")

🔍 Insights zur Paretoanalyse
----------------------------------------
📦 Ursprüngliche Lösungen:         979
🎯 Pareto-Kombinationen (2D):      2
🧩 Erweiterte Lösungskombis:       1958
✅ Nicht-dominierte Endlösungen:   38
❌ Entfernte (dominierte) Lösungen: 1920

📊 Pareto-Kombinationen (Anbaugeräte):
 Transport Attachments  Attachments
               1462.17         57.0
               4932.71         44.0

💾 Datei gespeichert unter: ParetoFront_filtered.csv
